# 🤖 Feedback Scoring Model — XGBoost Training

This notebook trains a regression model to predict interview answer quality (0–10) from NLP features.

**Features:**
- `semantic_score` — cosine similarity with ideal answer
- `keyword_score` — domain keyword coverage
- `clarity_score` — Flesch reading ease
- `confidence_score` — inverse filler word ratio

**Target:** `overall_score` (0–10, labeled by domain experts)

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import pickle
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded ✓')

In [ ]:
# Synthetic training data (replace with real labeled data)
# In production: collect from user interviews + expert ratings
np.random.seed(42)
n_samples = 500

# Simulate realistic feature distributions
semantic  = np.clip(np.random.beta(3, 2, n_samples), 0.1, 1.0)
keyword   = np.clip(np.random.beta(2, 2, n_samples), 0.0, 1.0)
clarity   = np.clip(np.random.beta(3, 3, n_samples), 0.1, 1.0)
confidence= np.clip(np.random.beta(4, 2, n_samples), 0.2, 1.0)

# Ground truth: weighted combo + small noise
scores = (0.35*semantic + 0.25*keyword + 0.25*clarity + 0.15*confidence) * 10
scores += np.random.normal(0, 0.4, n_samples)
scores = np.clip(scores, 0, 10)

X = np.column_stack([semantic, keyword, clarity, confidence])
y = scores

df = pd.DataFrame(X, columns=['semantic_score','keyword_score','clarity_score','confidence_score'])
df['overall_score'] = y
print(df.describe().round(2))

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f'MAE:  {mae:.3f} points (lower is better)')
print(f'R²:   {r2:.3f} (1.0 is perfect)')

In [ ]:
# Feature importance
feat_names = ['semantic_score','keyword_score','clarity_score','confidence_score']
importances = model.feature_importances_

plt.figure(figsize=(7,3))
bars = plt.barh(feat_names, importances, color=['#6366f1','#10b981','#f59e0b','#ef4444'])
plt.xlabel('Feature Importance')
plt.title('XGBoost Feature Importances')
plt.tight_layout()
plt.savefig('../docs/feature_importance.png', dpi=150)
plt.show()
print('Feature importance chart saved.')

In [ ]:
# Save model for production use
import os
os.makedirs('../data', exist_ok=True)
with open('../data/feedback_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Model saved to ../data/feedback_model.pkl ✓')

# Test on a new sample
test_input = np.array([[0.82, 0.65, 0.70, 0.88]])
pred = model.predict(test_input)[0]
print(f'Sample prediction: {round(pred, 1)}/10')